# FaceFinalML2 Colab Runner v3

Single-cell Colab runner with exact canonical mapping, official/pair-safe split, ArcFace/InsightFace embeddings, quantile-target regressors, and OOF stacking.

In [ ]:
# FaceFinalML2 Colab GPU runner: Face-to-BMI replication + modern embedding ensemble
# 1) In Colab: Runtime -> Change runtime type -> GPU, preferably A100.
# 2) Open this notebook from GitHub or upload it to Colab.
# 3) Run this single cell.
#
# This cell is self-contained. It clones/pulls the repo, downloads BMI.zip from
# Google Drive, audits the data, creates leakage-free slug/person splits, detects
# face crops, extracts frozen FaceNet/VGGFace2 + ConvNeXt + optional DINOv2
# embeddings, trains regularized regressors, evaluates against the paper's
# Pearson-r target, and writes outputs/metrics plus a Streamlit demo stub.
#
# It does not commit data or model artifacts to GitHub.

import os
import shlex
import subprocess
from pathlib import Path


def run(cmd, env=None, cwd=None):
    print(f"\n$ {cmd}", flush=True)
    p = subprocess.run(cmd, shell=True, env=env, cwd=cwd)
    if p.returncode != 0:
        raise SystemExit(f"Command failed with exit code {p.returncode}: {cmd}")


print("Checking GPU...")
gpu_check = subprocess.run("nvidia-smi", shell=True)
if gpu_check.returncode != 0:
    print("WARNING: No GPU is visible. In Colab, go to Runtime -> Change runtime type -> GPU, then rerun this cell.")

REPO = "manuelarceaguirre/facefinalml2"
DATA_FILE_ID = "16XA-MCnTG8ONdgxK0uPfFXWnA5oF3bFa"
WORKDIR = "/content/facefinalml2"
ZIP_PATH = f"{WORKDIR}/data/raw/BMI.zip"

if not os.path.isdir(WORKDIR):
    run(f"git clone https://github.com/{REPO}.git {shlex.quote(WORKDIR)}")
else:
    print(f"Repo already exists at {WORKDIR}; pulling latest...")
    run("git pull --ff-only || true", cwd=WORKDIR)

os.chdir(WORKDIR)

# Install a practical, Colab-friendly stack. InsightFace/ArcFace can be added later,
# but this runner avoids fragile native installs and uses facenet-pytorch + timm.
run("python -m pip install -q -U pip setuptools wheel")
run(
    "python -m pip install -q "
    "numpy pandas scipy scikit-learn xgboost joblib tqdm pillow matplotlib seaborn "
    "opencv-python-headless facenet-pytorch timm transformers accelerate "
    "gdown google-api-python-client insightface onnxruntime-gpu fastapi uvicorn python-multipart streamlit"
)

print("Authenticating to Google Drive for BMI.zip...")
print("Colab will open a Google sign-in/permission flow. Use the account that has access to the BMI.zip file.")
from google.colab import auth

auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/extracted").mkdir(parents=True, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    service = build("drive", "v3")
    meta = service.files().get(fileId=DATA_FILE_ID, fields="id,name,mimeType,size").execute()
    print(f"Signed-in Drive access confirmed: {meta}", flush=True)
    request = service.files().get_media(fileId=DATA_FILE_ID)
    with open(ZIP_PATH, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"Drive download: {int(status.progress() * 100)}%", flush=True)
else:
    print("BMI.zip already exists; skipping download.")

run("unzip -q -o data/raw/BMI.zip -d data/extracted")
run("find data/extracted -maxdepth 4 -type f | head -50")

os.environ.update({
    "PYTHONUNBUFFERED": "1",
    "CUDA_LAUNCH_BLOCKING": "0",
})

runner_code = '#!/usr/bin/env python3\n"""FaceFinalML2 Colab pipeline.\n\nThis script is written by notebooks/facefinalml2_colab_runner.ipynb inside Colab.\nIt performs the complete first-pass project pipeline:\n\n1. Audit BMI data and metadata.\n2. Build image-label rows.\n3. Create leakage-free group splits.\n4. Detect faces and write tight/loose crops.\n5. Extract frozen embeddings from FaceNet/VGGFace2, ConvNeXt, and optional DINOv2.\n6. Fit regularized regressors and a validation-weighted ensemble.\n7. Save metrics, predictions, plots, and a Streamlit demo stub.\n"""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport os\nimport random\nimport re\nimport time\nimport warnings\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Optional, Tuple\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image, ImageOps\nfrom scipy.stats import pearsonr, spearmanr\nfrom sklearn.decomposition import PCA\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.linear_model import ElasticNetCV, RidgeCV\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\nfrom sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler, QuantileTransformer\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.compose import TransformedTargetRegressor\nfrom sklearn.svm import SVR\nfrom sklearn.base import clone\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision import transforms\nfrom tqdm.auto import tqdm\n\nwarnings.filterwarnings("ignore", category=UserWarning)\n\nROOT = Path.cwd()\nDATA_ROOT = ROOT / "data"\nEXTRACTED = DATA_ROOT / "extracted"\nOUT = ROOT / "outputs"\nCROPS = OUT / "crops"\nFEATURES = OUT / "features"\nMETRICS = OUT / "metrics"\nMODELS = ROOT / "models"\nFIGURES = OUT / "figures"\n\nfor p in [OUT, CROPS, FEATURES, METRICS, MODELS, FIGURES]:\n    p.mkdir(parents=True, exist_ok=True)\n\nSEED = 42\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntorch.manual_seed(SEED)\nDEVICE = "cuda" if torch.cuda.is_available() else "cpu"\nprint(f"DEVICE={DEVICE}", flush=True)\n\nPAPER_BASELINE = {\n    "paper_vgg_net_svr_overall_r": 0.47,\n    "paper_vgg_face_svr_overall_r": 0.65,\n    "paper_vgg_face_male_r": 0.71,\n    "paper_vgg_face_female_r": 0.57,\n}\n\n\ndef norm_col(c: str) -> str:\n    return (\n        str(c).strip().lower()\n        .replace(" ", "_")\n        .replace("-", "_")\n        .replace("/", "_")\n        .replace("(", "")\n        .replace(")", "")\n        .replace("__", "_")\n    )\n\n\ndef pick_col(df: pd.DataFrame, candidates: Iterable[str], required: bool = False) -> Optional[str]:\n    columns = set(df.columns)\n    for c in candidates:\n        if c in columns:\n            return c\n    if required:\n        raise ValueError(f"Missing required column among {list(candidates)}; available={df.columns.tolist()}")\n    return None\n\n\ndef find_metadata_csv() -> Path:\n    csvs = sorted(EXTRACTED.rglob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)\n    if not csvs:\n        raise FileNotFoundError("No CSV found under data/extracted")\n    scored = []\n    for p in csvs:\n        try:\n            head = pd.read_csv(p, nrows=5)\n            cols = [norm_col(c) for c in head.columns]\n            score = 0\n            joined = " ".join(cols + [p.name.lower()])\n            for token in ["bmi", "weight", "height", "slug", "image", "photo"]:\n                score += int(token in joined)\n            scored.append((score, p.stat().st_size, p))\n        except Exception:\n            pass\n    if not scored:\n        return csvs[0]\n    scored.sort(reverse=True)\n    return scored[0][2]\n\n\ndef audit_and_build_rows() -> pd.DataFrame:\n    """Build the final image-label table using exact filename mapping.\n\n    The BMI zip used for this project contains a canonical metadata file with\n    columns like: unnamed:_0, bmi, gender, is_training, name. The previous v0\n    runner used fuzzy filename matching; this v2 runner intentionally avoids\n    fuzzy matching and maps each metadata row to BMI/Data/Images/<name>.\n    """\n    meta_path = find_metadata_csv()\n    print(f"Metadata CSV: {meta_path}", flush=True)\n    df = pd.read_csv(meta_path)\n    df = df.rename(columns={c: norm_col(c) for c in df.columns})\n    print(f"Raw metadata shape={df.shape}")\n    print(f"Columns={df.columns.tolist()}")\n    print(df.head().to_string())\n\n    name_col = pick_col(df, ["name", "filename", "file", "image", "img"], required=True)\n    bmi_col = pick_col(df, ["actual_bmi", "bmi", "body_mass_index"], required=True)\n    gender_col = pick_col(df, ["gender", "sex"])\n    is_training_col = pick_col(df, ["is_training", "training", "train", "split"])\n    row_id_col = pick_col(df, ["unnamed:_0", "unnamed_0", "index", "id"])\n\n    # Prefer the canonical image directory when present.\n    candidate_dirs = [\n        EXTRACTED / "BMI" / "Data" / "Images",\n        meta_path.parent / "Images",\n        meta_path.parent / "images",\n        EXTRACTED / "Images",\n        EXTRACTED / "images",\n    ]\n    img_dir = next((d for d in candidate_dirs if d.exists()), None)\n    if img_dir is None:\n        raise FileNotFoundError(f"Could not locate canonical Images directory. Tried: {candidate_dirs}")\n    print(f"Canonical image directory: {img_dir}", flush=True)\n\n    df["row_id"] = pd.to_numeric(df[row_id_col], errors="coerce").astype("Int64") if row_id_col else np.arange(len(df))\n    if df["row_id"].isna().any():\n        df["row_id"] = np.arange(len(df))\n    df["row_id"] = df["row_id"].astype(int)\n    df = df.sort_values("row_id").reset_index(drop=True)\n\n    # Hypothesis from VisualBMI: adjacent rows are before/after images for the same person.\n    df["pair_id"] = (df["row_id"].astype(int) // 2).astype(str)\n    df["group_id"] = df["pair_id"]\n    df["image_path"] = df[name_col].astype(str).map(lambda n: str(img_dir / n))\n    df["exists"] = df["image_path"].map(lambda p: Path(p).exists())\n\n    print("Exact path exists counts:")\n    print(df["exists"].value_counts(dropna=False).to_string())\n    if (~df["exists"]).any():\n        print("Missing exact image paths, first 20:")\n        print(df.loc[~df["exists"], ["row_id", name_col, bmi_col, "image_path"]].head(20).to_string(index=False))\n    df = df[df["exists"]].copy().reset_index(drop=True)\n\n    # Audit pair hypothesis and official split.\n    pair_audit = df.groupby("pair_id").agg(\n        n=(name_col, "count"),\n        gender_nunique=(gender_col, "nunique") if gender_col else (name_col, "count"),\n        split_nunique=(is_training_col, "nunique") if is_training_col else (name_col, "count"),\n        bmi_min=(bmi_col, "min"),\n        bmi_max=(bmi_col, "max"),\n    )\n    print("Pair audit summary:")\n    print(pair_audit.describe().to_string())\n    print("Pairs with n != 2:", int((pair_audit["n"] != 2).sum()))\n    if gender_col:\n        print("Pairs with gender mismatch:", int((pair_audit["gender_nunique"] != 1).sum()))\n    if is_training_col:\n        print("is_training counts:")\n        print(df[is_training_col].value_counts(dropna=False).to_string())\n        print("Pairs split across train/test:", int((pair_audit["split_nunique"] != 1).sum()))\n\n    df["bmi"] = pd.to_numeric(df[bmi_col], errors="coerce")\n    df = df[df["bmi"].between(13, 70)].copy()\n\n    if gender_col is not None:\n        df["gender_clean"] = df[gender_col].astype(str).str.lower().str.strip()\n    else:\n        df["gender_clean"] = "unknown"\n\n    if is_training_col is not None:\n        df["is_training_clean"] = pd.to_numeric(df[is_training_col], errors="coerce").fillna(-1).astype(int)\n    else:\n        df["is_training_clean"] = -1\n\n    def bmi_category(b: float) -> str:\n        if b < 18.5:\n            return "underweight"\n        if b < 25:\n            return "healthy"\n        if b < 30:\n            return "overweight"\n        if b < 35:\n            return "obesity_1"\n        if b < 40:\n            return "obesity_2"\n        return "obesity_3"\n\n    df["bmi_cat"] = df["bmi"].map(bmi_category)\n\n    ok, widths, heights = [], [], []\n    for p in tqdm(df["image_path"].tolist(), desc="verifying canonical images"):\n        try:\n            with Image.open(p) as im:\n                im.verify()\n            with Image.open(p) as im:\n                widths.append(im.size[0])\n                heights.append(im.size[1])\n            ok.append(True)\n        except Exception:\n            widths.append(np.nan)\n            heights.append(np.nan)\n            ok.append(False)\n    df["image_ok"] = ok\n    df["width"] = widths\n    df["height_px"] = heights\n    df = df[df["image_ok"]].copy().reset_index(drop=True)\n\n    print("Clean canonical rows:", df.shape)\n    print(df[["row_id", "pair_id", "image_path", "bmi", "bmi_cat", "gender_clean", "is_training_clean"]].head().to_string())\n    print("BMI summary:")\n    print(df["bmi"].describe().to_string())\n    print("Category counts:")\n    print(df["bmi_cat"].value_counts().to_string())\n\n    df.to_csv(OUT / "audit_image_rows_v2_canonical.csv", index=False)\n    pair_audit.to_csv(METRICS / "pair_audit_v2.csv")\n    return df\n\n\ndef make_splits(df: pd.DataFrame) -> pd.DataFrame:\n    """Create final evaluation splits.\n\n    If the dataset provides is_training, use it as the paper-comparable official\n    test split. Validation is carved only from the training pool with pair_id\n    grouping. Otherwise fall back to a pair-safe random split.\n    """\n    df = df.copy().reset_index(drop=True)\n    df["strata"] = df["gender_clean"].astype(str) + "_" + df["bmi_cat"].astype(str)\n    counts = df["strata"].value_counts()\n    df.loc[df["strata"].map(counts) < 5, "strata"] = "rare"\n\n    use_official = "is_training_clean" in df.columns and set(df["is_training_clean"].dropna().unique()).issuperset({0, 1})\n    if use_official:\n        print("Using official is_training split for test; validation is split from official training only.", flush=True)\n        train_pool = df[df["is_training_clean"] == 1].copy()\n        test = df[df["is_training_clean"] == 0].copy()\n        if len(test) == 0 or len(train_pool) == 0:\n            use_official = False\n        else:\n            # Grouped validation from the official training pool.\n            try:\n                sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)\n                train_idx, val_idx = next(sgkf.split(train_pool, train_pool["strata"], train_pool["group_id"]))\n                train = train_pool.iloc[train_idx].copy()\n                val = train_pool.iloc[val_idx].copy()\n            except Exception as e:\n                print(f"Official train StratifiedGroupKFold failed ({e}); using GroupShuffleSplit", flush=True)\n                gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)\n                train_idx, val_idx = next(gss.split(train_pool, groups=train_pool["group_id"]))\n                train = train_pool.iloc[train_idx].copy()\n                val = train_pool.iloc[val_idx].copy()\n\n    if not use_official:\n        print("Falling back to pair-safe random train/val/test split.", flush=True)\n        try:\n            sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)\n            trainval_idx, test_idx = next(sgkf.split(df, df["strata"], df["group_id"]))\n            trainval = df.iloc[trainval_idx].copy()\n            test = df.iloc[test_idx].copy()\n            sgkf2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + 1)\n            train_idx_rel, val_idx_rel = next(sgkf2.split(trainval, trainval["strata"], trainval["group_id"]))\n            train = trainval.iloc[train_idx_rel].copy()\n            val = trainval.iloc[val_idx_rel].copy()\n        except Exception as e:\n            print(f"StratifiedGroupKFold failed ({e}); falling back to GroupShuffleSplit", flush=True)\n            gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)\n            trainval_idx, test_idx = next(gss.split(df, groups=df["group_id"]))\n            trainval = df.iloc[trainval_idx].copy()\n            test = df.iloc[test_idx].copy()\n            gss2 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED + 1)\n            train_idx_rel, val_idx_rel = next(gss2.split(trainval, groups=trainval["group_id"]))\n            train = trainval.iloc[train_idx_rel].copy()\n            val = trainval.iloc[val_idx_rel].copy()\n\n    train["split"] = "train"\n    val["split"] = "val"\n    test["split"] = "test"\n    splits = pd.concat([train, val, test], ignore_index=True)\n\n    groups = {s: set(splits.loc[splits.split == s, "group_id"]) for s in ["train", "val", "test"]}\n    print("Group overlap audit:")\n    print("train-val overlap:", len(groups["train"] & groups["val"]))\n    print("train-test overlap:", len(groups["train"] & groups["test"]))\n    print("val-test overlap:", len(groups["val"] & groups["test"]))\n    assert groups["train"].isdisjoint(groups["val"])\n    assert groups["train"].isdisjoint(groups["test"])\n    assert groups["val"].isdisjoint(groups["test"])\n\n    split_path = OUT / "split_v2_official_or_pairsafe_seed42.csv"\n    splits.to_csv(split_path, index=False)\n    audit = splits.groupby("split").agg(\n        n_images=("image_path", "count"),\n        n_groups=("group_id", "nunique"),\n        bmi_mean=("bmi", "mean"),\n        bmi_std=("bmi", "std"),\n        bmi_min=("bmi", "min"),\n        bmi_max=("bmi", "max"),\n    )\n    print("Split audit:")\n    print(audit.to_string())\n    audit.to_csv(METRICS / "split_audit_v2.csv")\n    print("Category distribution by split:")\n    print(pd.crosstab(splits["split"], splits["bmi_cat"], normalize="index").round(3).to_string())\n    return splits\n\n\ndef square_expand_box(box, w: int, h: int, margin: float) -> Tuple[int, int, int, int]:\n    x1, y1, x2, y2 = [float(v) for v in box]\n    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0\n    side = max(x2 - x1, y2 - y1) * margin\n    nx1 = int(max(0, round(cx - side / 2)))\n    ny1 = int(max(0, round(cy - side / 2)))\n    nx2 = int(min(w, round(cx + side / 2)))\n    ny2 = int(min(h, round(cy + side / 2)))\n    if nx2 <= nx1 or ny2 <= ny1:\n        return 0, 0, w, h\n    return nx1, ny1, nx2, ny2\n\n\ndef center_square_box(w: int, h: int) -> Tuple[int, int, int, int]:\n    side = min(w, h)\n    x1 = (w - side) // 2\n    y1 = (h - side) // 2\n    return x1, y1, x1 + side, y1 + side\n\n\ndef make_crops(splits: pd.DataFrame) -> pd.DataFrame:\n    from facenet_pytorch import MTCNN\n\n    crop_csv = OUT / "split_v2_with_crops.csv"\n    if crop_csv.exists():\n        print(f"Using existing crops CSV: {crop_csv}")\n        return pd.read_csv(crop_csv)\n\n    tight_dir = CROPS / "tight_160"\n    loose_dir = CROPS / "loose_224"\n    tight_dir.mkdir(parents=True, exist_ok=True)\n    loose_dir.mkdir(parents=True, exist_ok=True)\n\n    mtcnn = MTCNN(keep_all=True, device=DEVICE)\n    rows = []\n    for idx, row in tqdm(splits.iterrows(), total=len(splits), desc="detecting/cropping faces"):\n        path = Path(row["image_path"])\n        image = Image.open(path).convert("RGB")\n        image = ImageOps.exif_transpose(image)\n        w, h = image.size\n        face_detected = False\n        det_score = 0.0\n        chosen_box = center_square_box(w, h)\n        try:\n            boxes, probs = mtcnn.detect(image)\n            if boxes is not None and len(boxes) > 0:\n                scores = []\n                for b, pr in zip(boxes, probs):\n                    x1, y1, x2, y2 = b\n                    area = max(1.0, (x2 - x1) * (y2 - y1)) / max(1.0, w * h)\n                    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2\n                    dist = math.sqrt(((cx - w / 2) / w) ** 2 + ((cy - h / 2) / h) ** 2)\n                    scores.append(float(pr or 0) + 0.25 * area - 0.10 * dist)\n                k = int(np.argmax(scores))\n                chosen_box = boxes[k]\n                det_score = float(probs[k] or 0)\n                face_detected = True\n        except Exception as e:\n            print(f"MTCNN failed for {path}: {e}")\n\n        tight_box = square_expand_box(chosen_box, w, h, margin=1.15) if face_detected else center_square_box(w, h)\n        loose_box = square_expand_box(chosen_box, w, h, margin=1.55) if face_detected else center_square_box(w, h)\n\n        stem = f"{idx:06d}_{re.sub(r\'[^A-Za-z0-9_.-]+\', \'_\', path.stem)[:80]}"\n        tight_path = tight_dir / f"{stem}.jpg"\n        loose_path = loose_dir / f"{stem}.jpg"\n\n        image.crop(tight_box).resize((160, 160), Image.BICUBIC).save(tight_path, quality=95)\n        image.crop(loose_box).resize((224, 224), Image.BICUBIC).save(loose_path, quality=95)\n\n        rec = row.to_dict()\n        rec.update({\n            "tight_crop_path": str(tight_path),\n            "loose_crop_path": str(loose_path),\n            "face_detected": bool(face_detected),\n            "det_score": float(det_score),\n            "tight_box": json.dumps([int(x) for x in tight_box]),\n            "loose_box": json.dumps([int(x) for x in loose_box]),\n        })\n        rows.append(rec)\n\n    out = pd.DataFrame(rows)\n    out.to_csv(crop_csv, index=False)\n    print(f"Face detection rate: {out[\'face_detected\'].mean():.3f}")\n    return out\n\n\nclass PathDataset(Dataset):\n    def __init__(self, paths: List[str], transform):\n        self.paths = list(paths)\n        self.transform = transform\n\n    def __len__(self):\n        return len(self.paths)\n\n    def __getitem__(self, idx):\n        img = Image.open(self.paths[idx]).convert("RGB")\n        return self.transform(img)\n\n\ndef extract_features(name: str, model: torch.nn.Module, paths: List[str], transform, batch_size: int = 96) -> np.ndarray:\n    cache = FEATURES / f"{name}.npy"\n    if cache.exists():\n        print(f"Loading cached features {cache}")\n        return np.load(cache)\n    ds = PathDataset(paths, transform)\n    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=(DEVICE == "cuda"))\n    feats = []\n    model = model.to(DEVICE).eval()\n    with torch.no_grad():\n        for x in tqdm(loader, desc=f"extracting {name}"):\n            x = x.to(DEVICE, non_blocking=True)\n            y = model(x)\n            if isinstance(y, (tuple, list)):\n                y = y[0]\n            feats.append(y.detach().float().cpu().numpy())\n    X = np.concatenate(feats, axis=0)\n    np.save(cache, X)\n    print(f"{name}: {X.shape}")\n    return X\n\n\n\ndef extract_arcface_features(df: pd.DataFrame) -> Tuple[Optional[np.ndarray], Optional[pd.DataFrame]]:\n    """Extract InsightFace ArcFace embeddings from the original canonical images.\n\n    This is the modern face-recognition branch intended to replace the broken\n    FaceNet branch. Missing detections are stored as NaNs and handled by\n    SimpleImputer inside regressors/stackers.\n    """\n    cache = FEATURES / "arcface_buffalo_l_original.npy"\n    meta_cache = FEATURES / "arcface_buffalo_l_original_meta.csv"\n    if cache.exists() and meta_cache.exists():\n        print(f"Loading cached ArcFace features {cache}")\n        return np.load(cache), pd.read_csv(meta_cache)\n\n    try:\n        import cv2\n        from insightface.app import FaceAnalysis\n    except Exception as e:\n        print(f"ArcFace skipped because insightface/cv2 import failed: {e}", flush=True)\n        return None, None\n\n    try:\n        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if DEVICE == "cuda" else ["CPUExecutionProvider"]\n        app = FaceAnalysis(name="buffalo_l", providers=providers)\n        app.prepare(ctx_id=0 if DEVICE == "cuda" else -1, det_size=(640, 640))\n    except Exception as e:\n        print(f"ArcFace buffalo_l initialization failed: {e}", flush=True)\n        return None, None\n\n    def pick_face(faces, img_shape):\n        h, w = img_shape[:2]\n        cx, cy = w / 2.0, h / 2.0\n        def score(face):\n            x1, y1, x2, y2 = face.bbox\n            area = max(0.0, x2 - x1) * max(0.0, y2 - y1)\n            fx, fy = (x1 + x2) / 2.0, (y1 + y2) / 2.0\n            center_dist = (((fx - cx) ** 2 + (fy - cy) ** 2) ** 0.5) / max(w, h)\n            return area * float(getattr(face, "det_score", 1.0)) - 0.15 * center_dist\n        return max(faces, key=score)\n\n    feats, rows = [], []\n    for p in tqdm(df["image_path"].tolist(), desc="extracting arcface_buffalo_l_original"):\n        img = cv2.imread(p)\n        if img is None:\n            feats.append(np.full(512, np.nan, dtype=np.float32))\n            rows.append({"arcface_detected": False, "arcface_n_faces": 0, "arcface_det_score": np.nan})\n            continue\n        try:\n            faces = app.get(img)\n        except Exception:\n            faces = []\n        if len(faces) == 0:\n            feats.append(np.full(512, np.nan, dtype=np.float32))\n            rows.append({"arcface_detected": False, "arcface_n_faces": 0, "arcface_det_score": np.nan})\n            continue\n        face = pick_face(faces, img.shape)\n        feats.append(face.normed_embedding.astype(np.float32))\n        rows.append({\n            "arcface_detected": True,\n            "arcface_n_faces": int(len(faces)),\n            "arcface_det_score": float(face.det_score),\n            "arcface_bbox_x1": float(face.bbox[0]),\n            "arcface_bbox_y1": float(face.bbox[1]),\n            "arcface_bbox_x2": float(face.bbox[2]),\n            "arcface_bbox_y2": float(face.bbox[3]),\n        })\n    X = np.vstack(feats)\n    meta = pd.DataFrame(rows)\n    np.save(cache, X)\n    meta.to_csv(meta_cache, index=False)\n    print(f"arcface_buffalo_l_original: {X.shape} detected={meta[\'arcface_detected\'].mean():.3f}")\n    return X, meta\n\ndef build_feature_sets(df: pd.DataFrame) -> Dict[str, np.ndarray]:\n    import timm\n    from facenet_pytorch import InceptionResnetV1, fixed_image_standardization\n    from timm.data import create_transform, resolve_data_config\n\n    feature_sets = {}\n\n    arc_X, arc_meta = extract_arcface_features(df)\n    if arc_X is not None:\n        feature_sets["arcface_buffalo_l_original"] = arc_X\n\n    facenet = InceptionResnetV1(pretrained="vggface2", classify=False).eval()\n    facenet_transform = transforms.Compose([\n        transforms.Resize((160, 160)),\n        transforms.ToTensor(),\n        fixed_image_standardization,\n    ])\n    feature_sets["facenet_vggface2_tight"] = extract_features(\n        "facenet_vggface2_tight",\n        facenet,\n        df["tight_crop_path"].tolist(),\n        facenet_transform,\n        batch_size=128,\n    )\n\n    conv_name_candidates = ["convnext_tiny.fb_in22k_ft_in1k", "convnext_tiny"]\n    conv_model = None\n    conv_name = None\n    for candidate in conv_name_candidates:\n        try:\n            conv_model = timm.create_model(candidate, pretrained=True, num_classes=0, global_pool="avg")\n            conv_name = candidate\n            break\n        except Exception as e:\n            print(f"Could not load {candidate}: {e}")\n    if conv_model is not None:\n        cfg = resolve_data_config({}, model=conv_model)\n        conv_transform = create_transform(**cfg)\n        feature_sets["convnext_loose"] = extract_features(\n            "convnext_loose",\n            conv_model,\n            df["loose_crop_path"].tolist(),\n            conv_transform,\n            batch_size=96,\n        )\n        print(f"Loaded ConvNeXt model: {conv_name}")\n\n    try:\n        dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", pretrained=True)\n        dino_transform = transforms.Compose([\n            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),\n            transforms.ToTensor(),\n            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),\n        ])\n        feature_sets["dinov2_vits14_loose"] = extract_features(\n            "dinov2_vits14_loose",\n            dino,\n            df["loose_crop_path"].tolist(),\n            dino_transform,\n            batch_size=96,\n        )\n    except Exception as e:\n        print(f"DINOv2 extraction skipped because loading failed: {e}", flush=True)\n\n    if len(feature_sets) >= 2:\n        ordered = [feature_sets[k] for k in sorted(feature_sets.keys())]\n        feature_sets["concat_all"] = np.concatenate(ordered, axis=1)\n        np.save(FEATURES / "concat_all.npy", feature_sets["concat_all"])\n        print(f"concat_all: {feature_sets[\'concat_all\'].shape}")\n\n    return feature_sets\n\n\ndef safe_corr(fn, y_true, y_pred) -> float:\n    y_true = np.asarray(y_true, dtype=float)\n    y_pred = np.asarray(y_pred, dtype=float)\n    if len(y_true) < 3 or np.std(y_true) == 0 or np.std(y_pred) == 0:\n        return float("nan")\n    try:\n        return float(fn(y_true, y_pred)[0] if fn is pearsonr else fn(y_true, y_pred).correlation)\n    except Exception:\n        return float("nan")\n\n\ndef regression_metrics(y_true, y_pred) -> Dict[str, float]:\n    y_true = np.asarray(y_true, dtype=float)\n    y_pred = np.asarray(y_pred, dtype=float)\n    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))\n    return {\n        "pearson_r": safe_corr(pearsonr, y_true, y_pred),\n        "spearman_rho": safe_corr(spearmanr, y_true, y_pred),\n        "mae": float(mean_absolute_error(y_true, y_pred)),\n        "rmse": rmse,\n        "r2": float(r2_score(y_true, y_pred)),\n        "bias": float(np.mean(y_pred - y_true)),\n        "within_2": float(np.mean(np.abs(y_pred - y_true) <= 2.0)),\n        "within_5": float(np.mean(np.abs(y_pred - y_true) <= 5.0)),\n    }\n\n\ndef make_regressors(n_features: int) -> Dict[str, object]:\n    base_ridge = make_pipeline(\n        SimpleImputer(strategy="mean"),\n        StandardScaler(),\n        RidgeCV(alphas=np.logspace(-4, 4, 41)),\n    )\n    regs = {\n        "ridge": clone(base_ridge),\n        "elastic": make_pipeline(\n            SimpleImputer(strategy="mean"),\n            StandardScaler(),\n            ElasticNetCV(\n                alphas=np.logspace(-4, 2, 25),\n                l1_ratio=[0.05, 0.1, 0.3, 0.5, 0.8],\n                max_iter=20000,\n                cv=5,\n                random_state=SEED,\n            ),\n        ),\n        "quantile_ridge": TransformedTargetRegressor(\n            regressor=clone(base_ridge),\n            transformer=QuantileTransformer(\n                n_quantiles=256,\n                output_distribution="normal",\n                random_state=SEED,\n            ),\n        ),\n        "pca_svr": make_pipeline(\n            SimpleImputer(strategy="mean"),\n            StandardScaler(),\n            PCA(n_components=min(256, max(2, n_features - 1)), random_state=SEED),\n            SVR(kernel="rbf", C=10.0, epsilon=0.3, gamma="scale"),\n        ),\n        "rf": make_pipeline(\n            SimpleImputer(strategy="mean"),\n            StandardScaler(),\n            RandomForestRegressor(n_estimators=400, max_depth=8, min_samples_leaf=3, random_state=SEED, n_jobs=-1),\n        ),\n    }\n    try:\n        from xgboost import XGBRegressor\n        regs["xgb"] = make_pipeline(\n            SimpleImputer(strategy="mean"),\n            StandardScaler(),\n            XGBRegressor(\n                n_estimators=700,\n                max_depth=3,\n                learning_rate=0.03,\n                subsample=0.85,\n                colsample_bytree=0.85,\n                objective="reg:squarederror",\n                random_state=SEED,\n                n_jobs=-1,\n            ),\n        )\n    except Exception as e:\n        print(f"XGBoost skipped: {e}")\n    return regs\n\n\ndef train_regressors(df: pd.DataFrame, feature_sets: Dict[str, np.ndarray]) -> Tuple[pd.DataFrame, List[dict]]:\n    y = df["bmi"].to_numpy(float)\n    train_mask = df["split"].eq("train").to_numpy()\n    val_mask = df["split"].eq("val").to_numpy()\n    test_mask = df["split"].eq("test").to_numpy()\n\n    records = []\n    fitted = []\n    for feat_name, X in feature_sets.items():\n        X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]\n        y_train, y_val, y_test = y[train_mask], y[val_mask], y[test_mask]\n        regs = make_regressors(X_train.shape[1])\n        for reg_name, reg in regs.items():\n            tag = f"{feat_name}__{reg_name}"\n            print(f"\\nTraining {tag} X={X_train.shape}", flush=True)\n            try:\n                reg.fit(X_train, y_train)\n                pred_val = reg.predict(X_val)\n                pred_test = reg.predict(X_test)\n                val_m = regression_metrics(y_val, pred_val)\n                test_m = regression_metrics(y_test, pred_test)\n                records.append({\n                    "model": tag,\n                    "feature_set": feat_name,\n                    "regressor": reg_name,\n                    "split": "val",\n                    **val_m,\n                })\n                records.append({\n                    "model": tag,\n                    "feature_set": feat_name,\n                    "regressor": reg_name,\n                    "split": "test",\n                    **test_m,\n                })\n                joblib.dump(reg, MODELS / f"{tag}.joblib")\n                fitted.append({\n                    "tag": tag,\n                    "feature_set": feat_name,\n                    "regressor": reg_name,\n                    "model": reg,\n                    "val_pred": pred_val,\n                    "test_pred": pred_test,\n                    "val_pearson": val_m["pearson_r"],\n                })\n                print(f"{tag}: val r={val_m[\'pearson_r\']:.4f} test r={test_m[\'pearson_r\']:.4f} MAE={test_m[\'mae\']:.3f}", flush=True)\n            except Exception as e:\n                print(f"FAILED {tag}: {e}", flush=True)\n\n    results = pd.DataFrame(records)\n    results.to_csv(METRICS / "model_results_long.csv", index=False)\n    if not results.empty:\n        wide = results.pivot_table(index="model", columns="split", values=["pearson_r", "mae", "rmse", "r2"], aggfunc="first")\n        wide.to_csv(METRICS / "model_results_wide.csv")\n        print("\\nTop validation models:")\n        print(results[results.split == "val"].sort_values("pearson_r", ascending=False).head(15).to_string(index=False))\n    return results, fitted\n\n\n\ndef oof_stacking_ensemble(df: pd.DataFrame, feature_sets: Dict[str, np.ndarray]) -> Optional[Dict[str, object]]:\n    """OOF positive-ish Ridge stacking on the official training pool.\n\n    Uses train+val as the official training pool, creates grouped OOF base\n    predictions, fits a Ridge stacker on OOF predictions, and reports official\n    test performance. This is more defensible than a single validation-weighted\n    blend.\n    """\n    pool_mask = df["split"].isin(["train", "val"]).to_numpy()\n    test_mask = df["split"].eq("test").to_numpy()\n    pool_df = df[pool_mask].reset_index(drop=True)\n    test_df = df[test_mask].reset_index(drop=True)\n    y_pool = pool_df["bmi"].to_numpy(float)\n    y_test = test_df["bmi"].to_numpy(float)\n\n    # Avoid collapsed FaceNet features unless they have real variance.\n    eligible = []\n    for name, X in feature_sets.items():\n        Xp = X[pool_mask]\n        finite_frac = float(np.isfinite(Xp).mean())\n        std = float(np.nanstd(Xp))\n        if finite_frac > 0.50 and std > 1e-6:\n            eligible.append(name)\n    print("OOF eligible feature sets:", eligible)\n    if not eligible:\n        return None\n\n    def ridge_builder(n_features):\n        return make_pipeline(SimpleImputer(strategy="mean"), StandardScaler(), RidgeCV(alphas=np.logspace(-4, 4, 41)))\n    def elastic_builder(n_features):\n        return make_pipeline(\n            SimpleImputer(strategy="mean"), StandardScaler(),\n            ElasticNetCV(alphas=np.logspace(-4, 2, 20), l1_ratio=[0.05, 0.1, 0.3, 0.5], max_iter=20000, cv=3, random_state=SEED),\n        )\n    def quantile_builder(n_features):\n        return TransformedTargetRegressor(\n            regressor=ridge_builder(n_features),\n            transformer=QuantileTransformer(n_quantiles=min(512, len(y_pool)), output_distribution="normal", random_state=SEED),\n        )\n    builders = {"ridge": ridge_builder, "elastic": elastic_builder, "quantile_ridge": quantile_builder}\n\n    base_names = [f"{feat}__{model}" for feat in eligible for model in builders]\n    oof = np.zeros((len(pool_df), len(base_names)), dtype=np.float32)\n    test_fold_preds = np.zeros((len(test_df), len(base_names), 5), dtype=np.float32)\n    strata = (pool_df["gender_clean"].astype(str) + "_" + pool_df["bmi_cat"].astype(str)).to_numpy()\n    groups = pool_df["group_id"].to_numpy()\n    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)\n\n    for fold, (tr_idx, va_idx) in enumerate(sgkf.split(pool_df, strata, groups)):\n        print(f"OOF fold {fold+1}/5", flush=True)\n        col = 0\n        for feat in eligible:\n            X_pool = feature_sets[feat][pool_mask]\n            X_test = feature_sets[feat][test_mask]\n            for model_name, builder in builders.items():\n                model = builder(X_pool.shape[1])\n                try:\n                    model.fit(X_pool[tr_idx], y_pool[tr_idx])\n                    oof[va_idx, col] = model.predict(X_pool[va_idx])\n                    test_fold_preds[:, col, fold] = model.predict(X_test)\n                except Exception as e:\n                    print(f"OOF base model failed {feat}__{model_name}: {e}")\n                    oof[va_idx, col] = np.nan\n                    test_fold_preds[:, col, fold] = np.nan\n                col += 1\n\n    keep = np.isfinite(oof).all(axis=0) & np.isfinite(test_fold_preds).all(axis=(0, 2))\n    base_names_keep = [n for n, k in zip(base_names, keep) if k]\n    oof = oof[:, keep]\n    test_base = test_fold_preds[:, keep, :].mean(axis=2)\n    if oof.shape[1] == 0:\n        return None\n\n    stacker = make_pipeline(SimpleImputer(strategy="mean"), StandardScaler(), RidgeCV(alphas=np.logspace(-4, 4, 41)))\n    stacker.fit(oof, y_pool)\n    oof_pred = stacker.predict(oof)\n    test_pred = stacker.predict(test_base)\n    oof_metrics = regression_metrics(y_pool, oof_pred)\n    test_metrics = regression_metrics(y_test, test_pred)\n\n    pred_df = test_df[["image_path", "group_id", "bmi", "bmi_cat", "gender_clean", "face_detected", "det_score"]].copy()\n    pred_df["pred_bmi"] = test_pred\n    pred_df["abs_error"] = (pred_df["pred_bmi"] - pred_df["bmi"]).abs()\n    pred_df.to_csv(METRICS / "oof_stacking_test_predictions.csv", index=False)\n\n    payload = {\n        "base_models": base_names_keep,\n        "oof_trainpool": oof_metrics,\n        "official_test": test_metrics,\n        "note": "OOF stacker trained only on official training pool (train+val); official test used for reporting.",\n    }\n    with open(METRICS / "oof_stacking_summary.json", "w") as f:\n        json.dump(payload, f, indent=2, sort_keys=True)\n    print("\\nOOF STACKING SUMMARY")\n    print(json.dumps(payload, indent=2, sort_keys=True))\n    return payload\n\n\ndef ensemble_and_subgroups(df: pd.DataFrame, fitted: List[dict]) -> Dict[str, object]:\n    val_df = df[df.split == "val"].reset_index(drop=True)\n    test_df = df[df.split == "test"].reset_index(drop=True)\n    y_val = val_df["bmi"].to_numpy(float)\n    y_test = test_df["bmi"].to_numpy(float)\n\n    usable = [m for m in fitted if np.isfinite(m["val_pearson"]) and m["val_pearson"] > 0]\n    usable = sorted(usable, key=lambda m: m["val_pearson"], reverse=True)[:8]\n    if not usable:\n        raise RuntimeError("No usable fitted models for ensemble")\n\n    weights = np.array([max(0.0, m["val_pearson"]) ** 2 for m in usable], dtype=float)\n    weights = weights / weights.sum()\n    val_pred = sum(w * m["val_pred"] for w, m in zip(weights, usable))\n    test_pred = sum(w * m["test_pred"] for w, m in zip(weights, usable))\n\n    payload = {\n        "paper_baseline": PAPER_BASELINE,\n        "ensemble_members": [\n            {"tag": m["tag"], "val_pearson": float(m["val_pearson"]), "weight": float(w)}\n            for w, m in zip(weights, usable)\n        ],\n        "ensemble_val": regression_metrics(y_val, val_pred),\n        "ensemble_test": regression_metrics(y_test, test_pred),\n    }\n\n    pred_rows = []\n    for split_name, part_df, pred in [("val", val_df, val_pred), ("test", test_df, test_pred)]:\n        tmp = part_df[["image_path", "group_id", "bmi", "bmi_cat", "gender_clean", "face_detected", "det_score"]].copy()\n        tmp["split"] = split_name\n        tmp["pred_bmi"] = pred\n        tmp["abs_error"] = (tmp["pred_bmi"] - tmp["bmi"]).abs()\n        pred_rows.append(tmp)\n    pred_df = pd.concat(pred_rows, ignore_index=True)\n    pred_df.to_csv(METRICS / "ensemble_predictions.csv", index=False)\n\n    subgroup_records = []\n    for split_name, part in pred_df.groupby("split"):\n        for col in ["gender_clean", "bmi_cat", "face_detected"]:\n            for value, g in part.groupby(col):\n                if len(g) >= 5:\n                    subgroup_records.append({\n                        "split": split_name,\n                        "subgroup_col": col,\n                        "subgroup": str(value),\n                        "n": int(len(g)),\n                        **regression_metrics(g["bmi"], g["pred_bmi"]),\n                    })\n    subgroup_df = pd.DataFrame(subgroup_records)\n    subgroup_df.to_csv(METRICS / "ensemble_subgroup_metrics.csv", index=False)\n\n    with open(METRICS / "final_summary.json", "w") as f:\n        json.dump(payload, f, indent=2, sort_keys=True)\n\n    print("\\nFINAL ENSEMBLE SUMMARY")\n    print(json.dumps(payload, indent=2, sort_keys=True))\n    if not subgroup_df.empty:\n        print("\\nSubgroup metrics:")\n        print(subgroup_df.sort_values(["split", "subgroup_col", "subgroup"]).to_string(index=False))\n    return payload\n\n\ndef write_streamlit_demo() -> None:\n    demo = r\'\'\'\nimport streamlit as st\nfrom PIL import Image\n\nst.title("Face-to-BMI Educational Demo")\nst.warning(\n    "Academic demo only. This is not a medical diagnostic tool. "\n    "BMI-from-face prediction is noisy, biased, and privacy-sensitive. "\n    "Do not use it for health, employment, insurance, or personal judgments."\n)\n\nst.write(\n    "This repository\'s Colab runner trains the final ensemble and writes metrics under `outputs/metrics`. "\n    "For a production-quality demo, load the saved regressors and reuse the same crop/embedding functions "\n    "from the Colab runner. This stub is intentionally conservative so the report can show the interface."\n)\n\nuploaded = st.file_uploader("Upload a face image", type=["jpg", "jpeg", "png"])\ncamera = st.camera_input("Or take a webcam photo")\nsource = uploaded or camera\nif source is not None:\n    image = Image.open(source).convert("RGB")\n    st.image(image, caption="Input image", width=320)\n    st.info("Connect this UI to the exported model bundle after final model selection.")\n\'\'\'\n    path = OUT / "streamlit_demo_stub.py"\n    path.write_text(demo)\n    print(f"Wrote demo stub: {path}")\n\n\ndef main() -> None:\n    start = time.time()\n    df = audit_and_build_rows()\n    splits = make_splits(df)\n    crop_df = make_crops(splits)\n    feature_sets = build_feature_sets(crop_df)\n    results, fitted = train_regressors(crop_df, feature_sets)\n    oof_summary = oof_stacking_ensemble(crop_df, feature_sets)\n    summary = ensemble_and_subgroups(crop_df, fitted)\n    write_streamlit_demo()\n    elapsed = (time.time() - start) / 60.0\n    print(f"\\nDone in {elapsed:.1f} minutes")\n    print("Important outputs:")\n    print(f"  {OUT / \'split_seed42_with_crops.csv\'}")\n    print(f"  {METRICS / \'model_results_long.csv\'}")\n    print(f"  {METRICS / \'final_summary.json\'}")\n    print(f"  {METRICS / \'oof_stacking_summary.json\'}")\n    print(f"  {METRICS / \'ensemble_predictions.csv\'}")\n    print(f"  {METRICS / \'ensemble_subgroup_metrics.csv\'}")\n    print(f"  {OUT / \'streamlit_demo_stub.py\'}")\n    print("Paper target: overall Pearson r > 0.65 on the held-out leakage-free test split.")\n\n\nif __name__ == "__main__":\n    main()'

Path("run_face_bmi_pipeline_v3.py").write_text(runner_code)

print("\nStarting Face-to-BMI v3 ArcFace/OOF official/pair-safe Colab pipeline.")
print("This writes:")
print("  face_bmi_pipeline_v3.out")
print("  outputs/split_v2_with_crops.csv")
print("  outputs/metrics/model_results_long.csv")
print("  outputs/metrics/final_summary.json")
print("  outputs/metrics/oof_stacking_summary.json")
print("  outputs/metrics/ensemble_predictions.csv")
print("  outputs/streamlit_demo_stub.py")
print("It does not commit raw data or model artifacts.\n")

env = os.environ.copy()
with open("face_bmi_pipeline_v3.out", "a", buffering=1) as log:
    p = subprocess.Popen(
        ["bash", "-lc", "python3 run_face_bmi_pipeline_v3.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="")
        log.write(line)
    rc = p.wait()
    if rc != 0:
        raise SystemExit(f"Face-to-BMI pipeline failed with exit code {rc}")

print("\nDone. Download or inspect these from the Colab file browser if needed:")
print("  face_bmi_pipeline_v3.out")
print("  outputs/metrics/final_summary.json")
print("  outputs/metrics/model_results_long.csv")
print("  outputs/metrics/ensemble_predictions.csv")
print("  outputs/streamlit_demo_stub.py")

try:
    import json
    summary = json.loads(Path("outputs/metrics/final_summary.json").read_text())
    print("\nFINAL SUMMARY")
    print(json.dumps(summary, indent=2, sort_keys=True)[:12000])
except Exception as e:
    print(f"Could not print final JSON summary: {e}")
